# Advanced GPU Project 3 — Train a CNN on CIFAR-10
## PyTorch • Computer vision • GPU training from scratch

**Task:** Train a convolutional neural network (CNN) to classify 32×32 colour images into 10 categories.  
**Dataset:** CIFAR-10 via `torchvision`.  
**Runtime:** GPU strongly recommended.

Unlike the ViT notebook, this example trains a small CNN from scratch. It demonstrates GPU tensor movement, data augmentation, training loops, validation, checkpoints, and inference.

## 1. Verify GPU and import libraries

In [ ]:
import torch, torchvision, platform
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU mode will be slower; enable GPU in Colab.")

## 2. Load CIFAR-10 and apply augmentation

In [ ]:
%pip -q install -U scikit-learn matplotlib

In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_set = datasets.CIFAR10(root="./data", train=False, download=True, transform=eval_transform)

# Deterministic train/validation indices; validation subset uses eval transforms.
g = torch.Generator().manual_seed(42)
train_len = 45000
val_len = len(full_train) - train_len
train_subset, val_subset = random_split(full_train, [train_len, val_len], generator=g)

# Rebuild validation dataset with evaluation transform (no augmentation).
val_full = datasets.CIFAR10(root="./data", train=True, download=False, transform=eval_transform)
val_set = torch.utils.data.Subset(val_full, val_subset.indices)

classes = full_train.classes
print("Classes:", classes)
print("Train:", len(train_subset), "Validation:", len(val_set), "Test:", len(test_set))

## 3. Visualize sample images

In [ ]:
def denormalize(img):
    mean_t = torch.tensor(mean).view(3,1,1)
    std_t = torch.tensor(std).view(3,1,1)
    return (img * std_t + mean_t).clamp(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    img, label = full_train[idx]
    ax.imshow(denormalize(img).permute(1,2,0))
    ax.set_title(classes[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Create data loaders

In [ ]:
BATCH_SIZE = 128
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
print("Batches:", len(train_loader), len(val_loader), len(test_loader))

## 5. Define a CNN

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout(0.15),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout(0.20),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN(num_classes=len(classes)).to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Define training and evaluation functions

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

def run_epoch(loader, training=False):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    return total_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

## 7. Train the CNN on GPU

In [ ]:
EPOCHS = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, training=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_cifar10_cnn.pth")

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"train loss {train_loss:.4f}, train acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f}, val acc {val_acc:.3f}")

print("Best validation accuracy:", round(best_val_acc, 4))

## 8. Plot learning curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs validation loss")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs validation accuracy")
plt.legend()
plt.show()

## 9. Evaluate the best checkpoint on test data

In [ ]:
model.load_state_dict(torch.load("best_cifar10_cnn.pth", map_location=device))
test_loss, test_acc, y_true, y_pred = run_epoch(test_loader, training=False)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.3f}")
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, xticks_rotation=45, cmap="Blues"
)
plt.tight_layout()
plt.show()

## 10. Predict one test image and save model

In [ ]:
images, labels = next(iter(test_loader))
image = images[0:1].to(device)
actual = classes[int(labels[0])]
model.eval()
with torch.no_grad():
    logits = model(image)
    probs = torch.softmax(logits, dim=1)[0]
pred_idx = int(probs.argmax())

plt.imshow(denormalize(images[0]).permute(1,2,0))
plt.axis("off")
plt.title(f"Predicted: {classes[pred_idx]} | Actual: {actual}")
plt.show()
print("Confidence:", f"{probs[pred_idx].item():.1%}")

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": classes,
    "mean": mean,
    "std": std
}, "cifar10_cnn_checkpoint.pth")
print("Saved cifar10_cnn_checkpoint.pth")

## Challenge extensions
- Add mixed-precision training with `torch.amp`.
- Compare CNN with a pretrained ResNet18.
- Train for more epochs and plot the learning curves.
- Add early stopping and experiment tracking.
- Analyze per-class errors and test-time confidence.

**Submit:** notebook, test accuracy and per-class metrics, learning curves, confusion matrix, model checkpoint, training time, GPU type, and limitations.